In [33]:
def print_bio_tags(bio_tags):
  for key, value in bio_tags.items():
        if value:
            print(f"{key}: {value}")

In [61]:
import re
import spacy

def update_phrase_list(sentence, phrase_list):
    res=phrase_list
    sentence_words = sentence.split()
    n=[]
    for p in phrase_list:
      l=p.split()
      n+=l

    longest_word=""
    comp=[]
    for word in sentence_words:
      if sentence_words.count(word) < n.count(word):
        for l in phrase_list:
          if l.startswith(word):
            comp.append(l)
        if comp:
          res.remove(min(comp, key=len))
          comp=[]

    return res

def extract_street_names(sentence):
    nlp = spacy.load("en_core_web_sm")
    stopwords = {"to", "from", "near", "at", "on", "in", "and"}
    regex_pattern = r'\b(?:to|from|near|at|on|in)\s((?:[a-z]+\s)*(?:\d+)|(?=\s\b(?:to|from|near|at|on|in|and)\b))'
    regex_matches = re.findall(regex_pattern, sentence)

    refined_matches = []
    for match in regex_matches:
        words = match.strip().split()
        stopword_indices = [i for i, word in enumerate(words) if word.lower() in stopwords]
        if stopword_indices:
            refined_words = []
            start_idx = 0
            for idx in stopword_indices:
                refined_words.append(' '.join(words[start_idx:idx]))
                start_idx = idx + 1

            refined_words.append(' '.join(words[start_idx:]))
            refined_matches = refined_words
        else:
            refined_matches.append(match.strip())

    doc = nlp(sentence)
    ner_matches = [ent.text for ent in doc.ents if ent.label_ in {"LOC", "GPE", "FAC"}]

    lst = update_phrase_list(sentence, list(set(refined_matches + ner_matches)))
    return lst

In [62]:
location_phrases = [
    # Current location phrases
    "your current location",
    "your location",
    "where you are",
    "where you are now",
    "your position",
    "current position",
    "right here",
    "this spot",
    "this location",
    "your present location",
    "current place",
    "this place",
    "where you are standing",
    "where you are at",

    # Immediate vicinity phrases
    "near you",
    "around you",
    "around here",
    "this area",
    "this neighborhood",
    "your vicinity",
    "nearby",
    "in this area",
    "from where you are",

    # GPS related
    "your gps location",
    "your coordinates",
    "your exact location",
    "your current coordinates",
    "your exact position",

    # Colloquial
    "where you are located",
    "your current spot",
    "this point",
    "your whereabouts",
    "current whereabouts",
    "present position",
    "right where you are",
    "this current location"
]

In [63]:
import re
import string

def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'\bmy\b', "your", sentence)
    sentence = re.sub(r'\bi am\b', "you are", sentence)
    sentence = re.sub(r'\bim\b', "you are", sentence)
    sentence = re.sub(r'\bi\b', "you", sentence)
    sentence = re.sub(r'\bme\b', "you", sentence)
    sentence = re.sub(r'\bfrom here\b', "from where you are", sentence)
    sentence = re.sub(r'\bto here\b', "to where you are", sentence)
    sentence = re.sub(r'\bwant to\b', "want", sentence)
    sentence = re.sub(r'\bto run\b', "run", sentence)
    sentence = re.sub(r'\bto explore\b', "explore", sentence)
    sentence = re.sub(r'\bto generate\b', "generate", sentence)
    sentence = re.sub(r'\bto make\b', "make", sentence)
    sentence = re.sub(r'\bto recive\b', "recive", sentence)
    sentence = sentence.replace(" and ", " ")
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    sentence = re.sub(r'(\d+)([a-zA-Z]+)', r'\1 \2', sentence)
    sentence = re.sub(r'(\d+)(\.\s)', r'\1 ', sentence)
    sentence = sentence.replace("\'", "")
    chars_to_replace = "-—–,;:!?'\"()[]{}@#$%^&*_+/|<>\\"
    sentence = sentence.translate(str.maketrans(chars_to_replace, ' ' * len(chars_to_replace)))
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    sentence = sentence.translate(str.maketrans('', '', string.punctuation.replace('.', '')))
    return sentence

In [64]:
difficulty_keywords = {
    "easy",
    "moderate",
    "hard",
    "beginner",
    "intermediate",
    "advanced",
    "simple",
    "basic",
    "challenging",
    "difficult",
    "entrylevel",
    "novice",
    "expert",
    "strenuous",
    "elementary",
    "complex",
    "very",
    "difficult"
}

In [65]:
def fill_difficulty(sentence, bio_tags):
  sucssus=False
  sentence_1=sentence
  tokens = sentence.replace('.', '').split()
  bio_tags['B-difficulty']=[]
  bio_tags['I-difficulty']=[]
  for i, token in enumerate(tokens):
    if token in difficulty_keywords and not sucssus:
      sucssus=True
      bio_tags['B-difficulty'].append(token)
    elif token in difficulty_keywords and sucssus:
      bio_tags['I-difficulty'].append(token)
  return sucssus, sentence_1, bio_tags

def append_to_start_location(st, bio_tags):
  full_st=st.split()
  if len(full_st) > 0:
    if full_st[-1].isdigit():
      if full_st[-1] in bio_tags['B-loca_start_num'] or full_st[-1] in bio_tags['B-loca_end_num']:
        return bio_tags
      bio_tags['B-loca_start_num'].append(full_st[-1])
      full_st=full_st[:-1]
    for word in full_st[1:]:
      bio_tags['I-start_location'].append(word)
  bio_tags['B-start_location'].append(full_st[0])
  return bio_tags

def append_to_end_location(st, bio_tags):
  full_st=st.split()
  if len(full_st)>0:
    if full_st[-1].isdigit():
      if full_st[-1] in bio_tags['B-loca_start_num'] or full_st[-1] in bio_tags['B-loca_end_num']:
        return bio_tags
      bio_tags['B-loca_end_num'].append(full_st[-1])
      full_st=full_st[:-1]
    for word in full_st[1:]:
      bio_tags['I-end_location'].append(word)
  bio_tags['B-end_location'].append(full_st[0])
  return bio_tags

def fill_start_location(sentence, bio_tags):
  sucssus=False
  bio_tags['B-start_location']=[]
  bio_tags['I-start_location']=[]
  bio_tags['B-loca_start_num']=[]
  sentence = preprocess_sentence(sentence)

  if any(item == sentence for item in location_phrases):
    location, _ = curr_location(sentence)
    bio_tags=append_to_start_location(location, bio_tags)
    sentence = sentence.replace(location, "")
    sucssus=True
    sentence, tokens = preprocess_sentence(sentence)
    return sucssus, sentence, bio_tags

  sucssus, _, bio_tags = fill_locations(1, sentence, bio_tags)
  location, _ = curr_location(sentence)
  if location:
    sentence = " ".join(sentence.split())
    if sucssus or any(phrase+" "+location in sentence for phrase in ["ending at", "finishing at", "ends at", "finishes at", "to", "ending"]):
      bio_tags=append_to_end_location(location, bio_tags)
      sentence = sentence.replace(location, "")
      sucssus=True
    elif any(phrase+" "+location in sentence for phrase in ["starting at", "begin at", "start at", "beginning at", "from", "starting"]) and not bio_tags['B-start_location']:
      bio_tags=append_to_start_location(location, bio_tags)
      sentence = sentence.replace(location, "")
      sucssus=True
    sentence = preprocess_sentence(sentence)
  return sucssus, sentence, bio_tags

def fill_end_location(sentence, bio_tags):
  sucssus=False
  bio_tags['B-end_location']=[]
  bio_tags['I-end_location']=[]
  bio_tags['B-loca_end_num']=[]
  sentence = preprocess_sentence(sentence)

  if any(item == sentence for item in location_phrases):
    location, _ = curr_location(sentence)
    bio_tags=append_to_end_location(location, bio_tags)
    sentence = sentence.replace(location, "")
    sucssus=True
    sentence = preprocess_sentence(sentence)
    return sucssus, sentence, bio_tags

  _, sucssus, bio_tags = fill_locations(0, sentence, bio_tags)
  location, _ = curr_location(sentence)
  if location:
    sentence = " ".join(sentence.split())
    if sucssus or any(phrase+" "+location in sentence for phrase in ["starting at", "begin at", "start at", "beginning at", "from", "starting"]):
      bio_tags=append_to_start_location(location, bio_tags)
      sentence = sentence.replace(location, "")
      sucssus=True
    elif any(phrase+" "+location in sentence for phrase in ["ending at", "finishing at", "ends at", "finishes at", "to", "ending"]) and not bio_tags['B-end_location']:
      bio_tags=append_to_end_location(location, bio_tags)
      sentence = sentence.replace(location, "")
      sucssus=True
    sentence = preprocess_sentence(sentence)
  return sucssus, sentence, bio_tags

def fill_route_length(sentence, bio_tags):
  sucssus=False
  bio_tags['B-route_length']=[]
  tokens=preprocess_sentence(sentence).split()
  if sentence.isdigit():
    sucssus=True
    bio_tags['B-route_length'].append(sentence)
  else:
    for i, token in enumerate(tokens):
      if token.isdigit():
        if i+1 < len(tokens) and tokens[i+1] in ["km", "kilometers", "meters", "miles", "k", "mile"]:
          sucssus=True
          bio_tags['B-route_length'].append(int(token))
      elif token.replace('.', '').isdigit():
        if i+1 < len(tokens) and tokens[i+1] in ["km", "kilometers", "meters", "miles", "k", "mile"]:
          sucssus=True
          bio_tags['B-route_length'].append(float(token))
  return sucssus, sentence, bio_tags

def fill_locations(start_end, sentence, bio_tags):
  start_sucssus=False
  end_sucssus=False
  ext_streets = extract_street_names(sentence)
  ext_streets = [item for item in ext_streets if not item.isdigit()]

  to_remove=[]
  if ext_streets:
    s= sentence.split()
    for st in ext_streets:
      for i, word in enumerate(s):
        if(i>0 and i<len(s)-1):
          if (word in["from", "starting"] or " ".join(s[i-1:i+1]) in ["starting at", "begin at", "beginning at", "start at", "starts at", "begins at"]) and " ".join(s[i + 1:]).startswith(st) and not bio_tags['B-start_location']:
            start_sucssus=True
            bio_tags=append_to_start_location(st, bio_tags)
            to_remove.append(st.split())
          elif (word in["to", "ending"] or " ".join(s[i-1:i+1]) in ["ending at", "finishing at", "ends at", "finishes at", "end at"]) and " ".join(s[i + 1:]).startswith(st) and not bio_tags['B-end_location']:
            end_sucssus=True
            bio_tags=append_to_end_location(st, bio_tags)
            to_remove.append(st)
  ext_streets = [item for item in ext_streets if item not in to_remove]
  if ext_streets and (not start_sucssus or end_sucssus):
    if start_end:
      for st in ext_streets:
        if start_end and not bio_tags['B-start_location'] and re.match(r"[a-z]+\s(?:[a-z]+\s)*\d+", sentence):
          start_sucssus=True
          bio_tags=append_to_start_location(st, bio_tags)
        elif not start_end and not bio_tags['B-end_location'] and re.match(r"[a-z]+\s(?:[a-z]+\s)*\d+", sentence):
          end_sucssus=True
          bio_tags=append_to_end_location(st, bio_tags)

  return start_sucssus, end_sucssus, bio_tags

def curr_location(sentence):
  for phrase in location_phrases:
    if phrase in sentence:
        return phrase, sentence.replace(phrase, "")
  return False, sentence

In [66]:
def complete_rest(sentence, bio_tags):
  i = 0
  sentence=preprocess_sentence(sentence)
  if not bio_tags['B-route_length']:
    _, sentence, bio_tags =fill_route_length(sentence, bio_tags)

  if not bio_tags['B-difficulty']:
    _, sentence, bio_tags = fill_difficulty(sentence, bio_tags)

  if not bio_tags['B-start_location']:
    _, sentence, bio_tags = fill_start_location(sentence, bio_tags)

  if not bio_tags['B-end_location']:
    _, sentence, bio_tags = fill_end_location(sentence, bio_tags)

  tokens = sentence.split()
  for token in tokens:
    if not any(token in values for values in bio_tags.values() if values != bio_tags['O']):
      bio_tags['O'].append(token)

  return False, tokens, bio_tags

In [67]:
def tag_bio(sentence, focus):
  focus_dict = {
      "start_loc": fill_start_location,
      "end_loc": fill_end_location,
      "diffculty_lvl": fill_difficulty,
      "route_lenght": fill_route_length,
  }

  bio_tags = {
    'B-loca_start_num': [],
    'I-start_location': [],
    'B-end_location': [],
    'I-difficulty': [],
    'B-route_length': [],
    'B-start_location': [],
    'I-route_length': [],
    'B-loca_end_num': [],
    'I-end_location': [],
    'B-difficulty': [],
    'O': []
  }
  sentence = preprocess_sentence(sentence)
  if focus in focus_dict:
    _, sentence, bio_tags = focus_dict[focus](sentence, bio_tags)
    _, _, bio_tags = complete_rest(sentence, bio_tags)
    # sentence = remove_bio_tags_words(bio_tags, sentence)
  elif focus == "all":
    _, _, bio_tags = complete_rest(sentence, bio_tags)

  return bio_tags

In [68]:
running_route_requests_1 = [
    "Looking for a 5km running route in the city center",
    "Need a 10 mile trail route for weekend training",
    "Where can I find a 3km beginner loop near the park?",
    "Searching for a half marathon (21.1km) training route",
    "Need a short 2 mile running path for lunch break",
    "Looking for an 8km scenic route along the river",
    "Can anyone recommend a 15km trail route with hills?",
    "Need a 1 mile running track for speed work",
    "Looking for a 30km long distance training route",
    "Where's a good 6 mile loop for morning runs?",
    "Need a 12km running route with good lighting",
    "Searching for a 4 mile coastal running path",
    "Looking for a 25km route for marathon training",
    "Need a quick 2.5km route near the office",
    "Where can I find a 7 mile suburban running loop?"
]

running_route_requests_2 = [
    "i want to generate a route. 10 km long, starting at my location and very hard. should end at haneviim 37. lets go!",
  "Need a running route from where im at",
   "Looking for a running route starting at asdfasdf 12",
   "Need a trail route beginning at my current location",
   "Running path from Downtown Station 19",
   "Route starting at Beach Boulevard Pier 5",
   "Looking for a run from here",
   "Running route from from where i am",
   "just go to here",
   "where im from we do not talk like that",
   "i want to run to explore the area",
   "to explore the aera from hanevim 12",
   "Looking for a run starting at asdf 09",
   "Trail route beginning at ghgf 18A and ending at my current location",
   "Trail route beginning at ghgf 18A to helo world 155",
]

running_route_requests_3 = [
   "Looking for a path finishing at my current location",
   "Need a running route that ends at asdfasdf 12",
   "Trail route ending at Mountain Peak lookout 14",
   "Running path that finishes at Downtown Square 55",
   "Need a route ending at University Sports Complex 43 from here",
   "Looking for a run that ends at River Street Bridge 22 from handd 32",
   "Need a running route to where im at"
]

running_route_requests_4 = [
    "Looking for a very easy running route near downtown",
    "Need a challenging trail run with hills",
    "Recommend a beginner-friendly 5k route",
    "Where can I find a very difficult mountain running path",
    "Searching for a moderate running loop in the park",
    "Need an intermediate route for my morning run",
    "Looking for a strenuous trail with elevation gain",
    "Simple flat running path for recovery day",
    "Advanced technical trail running route needed",
    "Basic running track for starting out",
    "Complex trail system for experienced runners",
    "Expert-level mountain running course",
    "Novice runner seeking entry-level path",
    "Elementary running loop for beginners",
    "Hard. trail run with technical sections",
    "no i want to start at my location",
    "yes but i want to run 5 km"
]

sentences = running_route_requests_1 + running_route_requests_2 + running_route_requests_3 + running_route_requests_4

In [69]:
focos=["start_loc", "end_loc", "diffculty_lvl", "route_lenght", "all"]
for i in range(0, 1): ############################################################################len(sentences) for full testings but takes a while!
  print("###"*100)
  print(sentences[i])
  print("###"*100)
  for focus in focos:
    print(" "*50, focus)
    bio_tags = tag_bio(sentences[i], focus)
    print_bio_tags(bio_tags)
    print("---"*100)

############################################################################################################################################################################################################################################################################################################
Looking for a 5km running route in the city center
############################################################################################################################################################################################################################################################################################################
                                                   start_loc
B-route_length: [5]
O: ['looking', 'for', 'a', '5', 'km', 'running', 'route', 'in', 'the', 'city', 'center']
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------